## 🧠 1️⃣ Import Libraries and Connect to SQLite ##

In [1]:
import sqlite3
import pandas as pd 

#Create a databse connection 
conn= sqlite3.connect("company_insights.db")

print("✅ Database connection successful!")

✅ Database connection successful!


## 🧾 2️⃣ Load CSV Files into pandas DataFrames ##

In [4]:
employees= pd.read_csv("employees.csv")
sales= pd.read_csv("sales.csv")
departments = pd.read_csv("departments.csv")

print("✅ CSV files loaded succesfully!")

✅ CSV files loaded succesfully!


## 💾 3️⃣ Store the DataFrames as SQL Tables ##

In [5]:
employees.to_sql("employees",conn,if_exists="replace", index= False)
sales.to_sql("sales",conn,if_exists="replace", index= False)
departments.to_sql("departments",conn,if_exists="replace", index= False)

print("✅ Tables created in SQL database!")

✅ Tables created in SQL database!


## 🔍 4️⃣ Verify Tables ##

In [6]:
query = "SELECT name FROM sqlite_master WHERE type='table';"
tables = pd.read_sql_query(query, conn)
print("📋 Tables in database:")
print(tables)


📋 Tables in database:
          name
0    employees
1        sales
2  departments


## 📄 5️⃣ Preview Each Table (Optional but Important) ##

In [8]:
print("EMPLOYEES:")
print(pd.read_sql_query("""SELECT 
* 
FROM employees
LIMIT 5 ;""",conn))

print("\nSALES:")
print(pd.read_sql_query("""SELECT 
* 
FROM sales
LIMIT 5 ;""",conn))

print("\nDEPARTMENTS:")
print(pd.read_sql_query("""SELECT 
* 
FROM departments
LIMIT 5 ;""",conn))

EMPLOYEES:
   EmployeeID      Name Department             Role       City Gender  Salary  \
0           1  Shreya_1      Sales  Sales Executive  Bangalore      F   55800   
1           2  Sameer_2      Sales  Sales Associate  Bangalore      M  116800   
2           3  Ananya_3      Sales  Sales Executive      Noida      M   77200   
3           4   Naman_4  Marketing   SEO Specialist      Noida      F   84100   
4           5   Parul_5         HR       HR Manager     Mumbai      F   81100   

     HireDate  PerformanceScore  ManagerID  Experience  
0  2017-09-30               4.9          4           3  
1  2021-08-15               3.3          7           1  
2  2021-04-17               3.9          4          12  
3  2021-08-10               3.3          5           1  
4  2018-02-12               3.0          3           4  

SALES:
   OrderID  EmployeeID  CustomerName Region         Category    Sales  Profit  \
0     1001          35   Neha Kapoor   West         Software  4637.96  

## 🧩 6️⃣ Verify Relationships (Joins Preview) ##

In [13]:
query = """
SELECT e.Name,
e.Department,
s.Category,
s.Sales,
d.Manager
FROM employees  AS e
JOIN sales AS s ON e.EmployeeID = s.EmployeeID
JOIN departments AS d ON e.Department = d.Department
LIMIT 10;
"""

joined_preview = pd.read_sql_query(query,conn)
print(joined_preview)

       Name Department         Category    Sales Manager
0  Shreya_1      Sales      Electronics  8769.05    Ravi
1  Shreya_1      Sales        Furniture  4411.49    Ravi
2  Shreya_1      Sales        Furniture  7265.89    Ravi
3  Shreya_1      Sales        Furniture  8880.40    Ravi
4  Shreya_1      Sales  Office Supplies  3224.24    Ravi
5  Shreya_1      Sales  Office Supplies  5703.06    Ravi
6  Shreya_1      Sales  Office Supplies  8810.83    Ravi
7  Shreya_1      Sales         Services   807.64    Ravi
8  Shreya_1      Sales         Services  9385.43    Ravi
9  Sameer_2      Sales      Electronics  4406.86    Ravi


# Data Analysis using SQL + Python #

## 1️⃣ Average Salary & Performance by Department ##

In [18]:
query1="""
SELECT
e.Department,
ROUND(AVG(e.Salary),2) AS Avg_salary,
ROUND(AVG(e.PerformanceScore),2) AS Avg_performance,
COUNT(e.EmployeeID) AS Total_employees
FROM employees e
GROUP BY e.Department 
ORDER BY Avg_salary DESC;
"""
avg_salary_perf= pd.read_sql_query(query1,conn)
print("Average Salary & Performance by Department:")
print(avg_salary_perf)

Average Salary & Performance by Department:
  Department  Avg_salary  Avg_performance  Total_employees
0    Finance    84917.65             4.08               17
1         IT    84159.09             3.88               22
2      Sales    83395.24             3.94               21
3  Marketing    76643.48             4.13               23
4         HR    69358.82             3.91               17


## 2️⃣ Total Sales & Profit by Region ##

In [17]:
query2="""
SELECT 
   s.Region,
   ROUND(SUM(s.Sales),2) AS Total_sales ,
   ROUND(SUM(s.Profit),2) AS Total_profit ,
   ROUND(SUM(s.Profit)/ SUM(s.Sales)*100,2) AS Profit_Margin 
FROM sales s
GROUP BY s.Region
ORDER BY Total_sales DESC;
"""

region_sales = pd.read_sql_query(query2,conn)
print("\nTotal Sales & Profit by Region")
print(region_sales)


Total Sales & Profit by Region
  Region  Total_sales  Total_profit  Profit_Margin
0  South   1372223.31     203341.20          14.82
1  North   1332194.53     205570.19          15.43
2   East   1271727.34     191182.54          15.03
3   West   1197683.40     189922.19          15.86


## 3️⃣ Top 10 Employees by Total Sales ##

In [19]:
query3= """
SELECT
    e.Name,
    e.Department,
    ROUND(SUM(s.sales),2) AS Total_sales,
    ROUND(SUM(s.Profit),2) As Total_profits
FROM employees e
JOIN sales s ON e.EmployeeID = s.EmployeeID
GROUP BY e.EmployeeID
ORDER BY Total_sales DESC
LIMIT 10 ;
"""

top_employees= pd.read_sql_query(query3,conn)
print("\nTop 10 Employees by Total Sales")
print(top_employees)


Top 10 Employees by Total Sales
        Name Department  Total_sales  Total_profits
0   Manav_59         IT    111363.93       17177.25
1    Isha_95  Marketing     86546.22       12617.39
2  Sameer_41  Marketing     83551.60       13651.23
3    Amit_82         HR     82677.86       14136.71
4  Shruti_65  Marketing     78769.32       13380.64
5  Aditya_18  Marketing     78323.33       12915.86
6   Sonia_81         IT     75305.01       11154.48
7  Simran_38    Finance     74922.88       11079.99
8    Neha_90         IT     73556.54        9987.57
9  Aditya_36         HR     73139.27       10703.06


## 4️⃣ Department-wise Profit Contribution ##

In [20]:
query4= """
SELECT 
   e.Department,
   ROUND(SUM(s.Profit),2) AS Total_Profit,
   ROUND(SUM(s.Profit)/ (SELECT SUM(Profit) FROM sales)*100,2) AS Profit_share
FROM employees e
JOIN sales s ON e.EmployeeID= s.EmployeeID
GROUP BY e.Department 
ORDER BY Total_Profit DESC;
"""
dept_profit= pd.read_sql_query(query4,conn)
print("\nDepartment-wise Profit Contribution:")
print(dept_profit)


Department-wise Profit Contribution:
  Department  Total_Profit  Profit_share
0  Marketing     187657.45         23.75
1         IT     176930.16         22.40
2      Sales     162831.95         20.61
3         HR     135957.87         17.21
4    Finance     126638.69         16.03


## 5️⃣ Yearly Sales Trend ##

In [21]:
query5= """
SELECT
   strftime("%Y",s.Date) AS Year,
   ROUND(SUM(s.Sales),2) AS Total_sales,
   ROUND(SUM(s.Profit),2) AS Total_Profit
FROM sales s
GROUP BY Year
ORDER BY Year;
"""
sales_trend= pd.read_sql_query(query5,conn)
print("\nYearly Sales Trend:")
print(sales_trend)


Yearly Sales Trend:
   Year  Total_sales  Total_Profit
0  2020    976090.90     155065.05
1  2021    998731.79     146319.00
2  2022   1044814.83     157774.89
3  2023   1143544.39     179713.60
4  2024   1010646.67     151143.58


## 6️⃣ Discount vs Profit Relationship ##

In [23]:
query6= """
SELECT
  ROUND(s.Discount,2) AS Discount,
  ROUND(AVG(s.Profit),2) AS Avg_Profit
FROM sales s
GROUP BY Discount
ORDER BY Discount;
"""

discount_impact= pd.read_sql_query(query6,conn)
print("\nImpact of Discount on Profit:")
print(discount_impact)


Impact of Discount on Profit:
    Discount  Avg_Profit
0       0.00      752.68
1       0.01      863.76
2       0.02      834.27
3       0.03      863.22
4       0.04      854.40
5       0.05      793.02
6       0.06      716.95
7       0.07      710.39
8       0.08      637.23
9       0.09      849.54
10      0.10      796.28
11      0.11      674.69
12      0.12      857.11
13      0.13      821.38
14      0.14      690.85
15      0.15      812.38
16      0.16      736.88
17      0.17      906.85
18      0.18      815.76
19      0.19      763.03
20      0.20      837.36


In [24]:
avg_salary_perf.to_csv("avg_salary_perf.csv", index=False)
region_sales.to_csv("region_sales.csv", index=False)
top_employees.to_csv("top_employees.csv", index=False)
dept_profit.to_csv("dept_profit.csv", index=False)
sales_trend.to_csv("sales_trend.csv", index=False)
discount_impact.to_csv("discount_impact.csv", index=False)

print("✅ Data exported for Power BI successfully!")


✅ Data exported for Power BI successfully!
